# Clase 10 - Validación Cruzada

Veremos por qué no alcanza con un único split de train y test, usar
validación cruzada para comparar modelos y elegir hiperparámetros sin hacer
trampa con el test set, y cuidarnos del data leakage.

Usamos otra vez el dataset de cáncer de mama. Como vimos antes, invertimos las etiquetas para que `1 = maligno` sea la clase
positiva (la que queremos detectar).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = (cancer.target == 0).astype(int)   # 1 = maligno

## 1. ¿Cuánto vale un solo split?

Hasta ahora siempre separamos train y test una vez, con `random_state=0`.
¿Qué pasa si cambiamos solo ese número? Mismo modelo, mismos datos:

In [ ]:
accuracies = []
for semilla in range(30):
    X_train, X_test, y_train, y_test = train_test_split(
        X_cancer, y_cancer, test_size=0.2, random_state=semilla, stratify=y_cancer
    )
    arbol = DecisionTreeClassifier(max_depth=4, random_state=0)
    arbol.fit(X_train, y_train)
    accuracies.append(arbol.score(X_test, y_test))

print("Peor split: ", round(min(accuracies), 3))
print("Mejor split:", round(max(accuracies), 3))

plt.hist(accuracies, bins=10, edgecolor="k")
plt.xlabel("Accuracy en test")
plt.ylabel("Cantidad de splits")
plt.title("El mismo árbol, 30 splits distintos")
plt.show()

Según lo que toque, el mismo modelo saca entre 0.88 y 0.96. Si
comparamos dos modelos con un solo split, una diferencia de 3 o 4 puntos
puede ser pura suerte.

## 2. Validación cruzada

Partir los datos en `k` pedazos (folds) y entrenar `k` veces, cada
vez validando con un fold distinto. Veamos cómo reparte `KFold` 10 datos en
5 folds:

In [ ]:
from sklearn.model_selection import KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=0)
datos = np.arange(10)

for ronda, (idx_train, idx_val) in enumerate(kfold.split(datos), start=1):
    print(f"Ronda {ronda}: train = {datos[idx_train]}, validación = {datos[idx_val]}")

Cada dato aparece en validación exactamente una vez. Ahora lo hacemos con el dataset sobre cáncer de mama:

In [ ]:
scores = []
for idx_train, idx_val in kfold.split(X_cancer):
    arbol = DecisionTreeClassifier(max_depth=4, random_state=0)
    arbol.fit(X_cancer.iloc[idx_train], y_cancer.iloc[idx_train])
    scores.append(arbol.score(X_cancer.iloc[idx_val], y_cancer.iloc[idx_val]))

print("Scores de cada fold:", np.round(scores, 3))
print(f"Promedio: {np.mean(scores):.3f} +- {np.std(scores):.3f}")

`cross_val_score` hace todo eso en una línea. Con un clasificador, además,
arma los folds estratificados, cada uno con la misma proporción de
malignos que el dataset completo.

In [ ]:
from sklearn.model_selection import cross_val_score

arbol = DecisionTreeClassifier(max_depth=4, random_state=0)
scores = cross_val_score(arbol, X_cancer, y_cancer, cv=5)

print("Accuracy de cada fold:", scores.round(3))
print(f"Accuracy: {scores.mean():.3f} +- {scores.std():.3f}")

Con `scoring` elegimos otra métrica:

In [ ]:
recalls = cross_val_score(arbol, X_cancer, y_cancer, cv=5, scoring="recall")
print(f"Recall: {recalls.mean():.3f} +- {recalls.std():.3f}")

## 3. Elegir hiperparámetros

En la clase de árboles elegimos `max_depth` mirando el accuracy en test. Ahora lo
hacemos bien con las medialunas:

1. Separamos el test y no lo tocamos.
2. Elegimos `max_depth` con validación cruzada dentro del train.
3. Al final, una sola vez, evaluamos en test.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.35, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

profundidades = range(1, 16)
cv_medias, cv_desvios = [], []

for d in profundidades:
    scores = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0), X_train, y_train, cv=5)
    cv_medias.append(scores.mean())
    cv_desvios.append(scores.std())

cv_medias, cv_desvios = np.array(cv_medias), np.array(cv_desvios)
plt.plot(profundidades, cv_medias, marker="o", label="validación cruzada (promedio)")
plt.fill_between(profundidades, cv_medias - cv_desvios, cv_medias + cv_desvios, alpha=0.2, label="+- 1 desvío")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

mejor_d = profundidades[np.argmax(cv_medias)]
print("Mejor max_depth según validación cruzada:", mejor_d)

Elegimos la profundidad sin haber mirado el test. Recién ahora lo chequeamos:

In [ ]:
arbol_final = DecisionTreeClassifier(max_depth=mejor_d, random_state=0)
arbol_final.fit(X_train, y_train)
print("Accuracy en test:", round(arbol_final.score(X_test, y_test), 3))

### `GridSearchCV`: lo mismo pero automático

`GridSearchCV` prueba todas las combinaciones de una grilla de
hiperparámetros, cada una con validación cruzada, y al final reentrena el
mejor modelo con todo el train. Probemos kNN con distintos `k`.

Como kNN va dentro de un pipeline, el nombre del hiperparámetro lleva
adelante el nombre del paso: `kneighborsclassifier__n_neighbors` (con dos
guiones bajos).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

pipeline_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
grilla = {"kneighborsclassifier__n_neighbors": range(1, 60, 2)}

busqueda = GridSearchCV(pipeline_knn, grilla, cv=5)
busqueda.fit(X_train, y_train)

print("Mejor k:", busqueda.best_params_)
print("Accuracy en validación cruzada:", round(busqueda.best_score_, 3))
print("Accuracy en test:", round(busqueda.score(X_test, y_test), 3))

Los resultados de cada combinación quedan en `cv_results_`:

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_)
plt.errorbar(
    resultados["param_kneighborsclassifier__n_neighbors"].astype(int),
    resultados["mean_test_score"],
    yerr=resultados["std_test_score"],
    marker="o", capsize=3,
)
plt.xlabel("k")
plt.ylabel("Accuracy (validación cruzada)")
plt.show()

## 4. Data leakage

Un experimento raro. Generamos 100 pacientes con 5000 columnas de
ruido y una etiqueta 0/1 en algun lugar. No hay nada que
aprender: cualquier modelo interesante debería sacar alrededor de 50%.

In [ ]:
rng_ruido = np.random.default_rng(seed=0)
X_ruido = rng_ruido.normal(size=(100, 5000))
y_ruido = rng_ruido.integers(0, 2, size=100)

Con tantas columnas, por pura casualidad algunas van a estar algo
relacionadas con `y_ruido`. `SelectKBest` se queda con las 20 columnas más
relacionadas con la etiqueta.

La forma incorrecta es elegir las 20 columnas mirando todos los datos,
y después hacer validación cruzada.

In [ ]:
from sklearn.feature_selection import SelectKBest

X_elegidas = SelectKBest(k=20).fit_transform(X_ruido, y_ruido)   # mira TODOS los datos

scores = cross_val_score(LogisticRegression(), X_elegidas, y_ruido, cv=5)
print(f"Accuracy (con leakage): {scores.mean():.2f}")

Predice ruido muy bien! Lo que pasa es que al elegir las
columnas usamos las etiquetas de todos los datos, incluidos los que
después usamos para validar. Las columnas elegidas conocen la respuesta
de los datos de validación.

La forma correcta es poner la selección de features adentro de un
pipeline, así en cada ronda se elige mirando solo los folds de train.

In [ ]:
pipeline_honesto = make_pipeline(SelectKBest(k=20), LogisticRegression())

scores = cross_val_score(pipeline_honesto, X_ruido, y_ruido, cv=5)
print(f"Accuracy (sin leakage): {scores.mean():.2f}")

Alrededor de 50%, como tiene que ser. Sobre ruido no hay nada que aprender. Lo mismo pasa con el
`StandardScaler` y con cualquier paso que aprenda algo de los datos, todo
va adentro del pipeline.

## 5. Comparando modelos con validación cruzada

En la clase de kNN y SVM comparamos los cuatro clasificadores con un solo split.
Ahora lo hacemos bien, sobre cáncer de mama, y midiendo también el recall
(cuántos tumores malignos encuentra cada uno). `cross_validate` permite
pedir varias métricas a la vez.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.svm import SVC

modelos = {
    "Logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "Árbol (depth=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM (rbf)": make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}

filas = []
for nombre, modelo in modelos.items():
    cv = cross_validate(modelo, X_cancer, y_cancer, cv=5, scoring=["accuracy", "recall", "f1"])
    filas.append({
        "modelo": nombre,
        "accuracy": f"{cv['test_accuracy'].mean():.3f} +- {cv['test_accuracy'].std():.3f}",
        "recall": f"{cv['test_recall'].mean():.3f} +- {cv['test_recall'].std():.3f}",
        "F1": f"{cv['test_f1'].mean():.3f} +- {cv['test_f1'].std():.3f}",
    })

pd.DataFrame(filas).set_index("modelo")

Fíjense en los desvíos: si la diferencia entre dos modelos es mucho más
chica que el desvío, no podemos decir con confianza que uno sea mejor que el
otro.

## Ejercicios

**Ejercicio 1.** Usar `GridSearchCV` para buscar a la vez `max_depth` en
`[2, 3, 4, 5, 6, 8, None]` y `min_samples_leaf` en `[1, 5, 10, 20]` para un
árbol sobre las medialunas. ¿Cuántos entrenamientos hace en total? Comparar
el accuracy en test contra el árbol de la sección 3.

**Ejercicio 2.** Repetir el experimento de la sección 1 (30 splits distintos)
pero ahora con `cross_val_score` usando
`KFold(n_splits=5, shuffle=True, random_state=semilla)` como `cv`. Comparar
el histograma de los promedios contra el de los splits individuales. ¿Cuál
varía menos?

**Ejercicio 3.** Repetir el experimento de data leakage de la sección 4 con
100, 1000 y 5000 columnas de ruido. ¿Cómo cambia el accuracy "con leakage"?
¿Por qué?

**Ejercicio 4.** Agregar `RandomForestClassifier` (de `sklearn.ensemble`) al
duelo de la sección 5, y agregar `"precision"` a las métricas. Si tuvieran
que elegir un modelo para un hospital, ¿cuál elegirían y por qué?

In [ ]:
# Ejercicio 1: 

In [ ]:
# Ejercicio 2: 

In [ ]:
# Ejercicio 3: 

In [ ]:
# Ejercicio 4: 